In [1]:
import os
os.environ["GEMINI_API_KEY"] = "***"

In [2]:
!pip install langchain chromadb faiss-cpu tiktoken langchain_google-genai langchain-community wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━

# Wikipedia Retriever

In [3]:
from langchain_community.retrievers import WikipediaRetriever

In [4]:
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [5]:
query = "History of india and pakistan from perspective of china"
docs = retriever.invoke(query)

# Vector Store Retriever

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [7]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [9]:
vector_store = Chroma.from_documents(
    embedding = embeddings,
    documents = documents,
    collection_name = "my_collection"
)

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k":2})

In [11]:
query = "What is chroma used for?"
result = retriever.invoke(query)

In [12]:
for i, doc in enumerate(result):
    print(f"{i+1}. {doc.page_content}")

1. Chroma is a vector database optimized for LLM-based search.
2. Embeddings convert text into high-dimensional vectors.


# Maximal Marginal Relevance (MMR)

In [21]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [14]:
from langchain_community.vectorstores import FAISS

In [15]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [22]:
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embeddings
)

In [30]:
retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3, "lambda_mult":0}
)
# k = top results, lambda_mult = relevance diversity balance (0 or 1, 0 = highly diverse results)

In [31]:
query = "what is langchain"
results = retriever.invoke(query)

In [32]:
for i, doc in enumerate(results):
    print(f"{i+1}. {doc.page_content}")

1. LangChain is used to build LLM based applications.
2. MMR helps you get diverse results when doing similarity search.
3. Chroma is used to store and search document embeddings.


# Multi-Query Retriever

In [33]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import MultiQueryRetriever

In [34]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [35]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [36]:
vectorstore = FAISS.from_documents(
    documents = all_docs,
    embedding = embeddings
)

In [37]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [43]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":4}),
    llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
)

In [39]:
query="How to improve energy levels and maintain balance?"

In [44]:
sim_results = similarity_retriever.invoke(query)
multi_results = multiquery_retriever.invoke(query)

In [45]:
print("similarity_retriever_results")
for i, doc in enumerate(sim_results):
    print(f"{i+1}. {doc.page_content}")

similarity_retriever_results
1. Drinking sufficient water throughout the day helps maintain metabolism and energy.
2. The solar energy system in modern homes helps balance electricity demand.
3. Mindfulness and controlled breathing lower cortisol and improve mental clarity.
4. Consuming leafy greens and fruits helps detox the body and improve longevity.
5. Regular walking boosts heart health and can reduce symptoms of depression.


In [46]:
print("multiquery_retriever_results")
for i, doc in enumerate(multi_results):
    print(f"{i+1}. {doc.page_content}")

multiquery_retriever_results
1. Drinking sufficient water throughout the day helps maintain metabolism and energy.
2. Consuming leafy greens and fruits helps detox the body and improve longevity.
3. Mindfulness and controlled breathing lower cortisol and improve mental clarity.
4. Regular walking boosts heart health and can reduce symptoms of depression.
5. The solar energy system in modern homes helps balance electricity demand.


# Contextual Compression Retriever

In [47]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor

In [ ]:
# docs
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [48]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [49]:
vectorstore = FAISS.from_documents(
    documents = all_docs,
    embedding = embeddings
)

In [50]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [51]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [52]:
compressor = LLMChainExtractor.from_llm(llm)

In [53]:
retriever = ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = base_retriever
)

In [54]:
query = "what is photosynthesis"
results = retriever.invoke(query)

In [55]:
for i, doc in enumerate(results):
    print(f"{i+1}. {doc.page_content}")

1. Photosynthesis enables plants to produce energy by converting sunlight.
